# VisionExtractor (OpenRouter) Test Notebook

This notebook demonstrates and tests the **VisionExtractor** strategy (Strategy C)
which uses **OpenRouter-hosted vision models** (e.g., GPT-4o-mini, Gemini Pro Vision)
for high-fidelity extraction:

- Scanned PDFs and OCR-heavy documents
- Handwriting and degraded layouts
- Complex multi-column layouts with tables/figures
- Budget-aware extraction with automatic cost tracking

> Prerequisites:
> - `OPENROUTER_API_KEY` environment variable set
> - `pdf2image` installed: `pip install pdf2image`
> - `poppler` installed on your system (for pdf2image)
> - OpenRouter account with credits


In [ ]:
# Setup and Imports

import os
import sys
from pathlib import Path

# Check for OPENROUTER_API_KEY
api_key = os.getenv("OPENROUTER_API_KEY", "")
if not api_key:
    print("⚠️ WARNING: OPENROUTER_API_KEY not set. VisionExtractor will fail at runtime.")
    print("   Set it with: os.environ['OPENROUTER_API_KEY'] = 'your-key-here'")
else:
    print(f"✓ OPENROUTER_API_KEY found (length: {len(api_key)})")

# Add src to path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies.vision_augmented import VisionExtractor
from src.models.document_profile import DocumentProfile


In [ ]:
# Select Test Document

DATA_ROOT = Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data")

TEST_DOCS = {
    "class_a": DATA_ROOT / "class_a" / "CBE_ANNUAL_REPORT_2023-24.pdf",
    "class_b": DATA_ROOT / "class_b" / "Annual_Report_JUNE-2023.pdf",
    "class_c": DATA_ROOT / "class_c" / "fta_performance_survey_final_report_2022.pdf",
    "class_d": DATA_ROOT / "class_d" / "tax_expenditure_ethiopia_2021_22.pdf",
}

# For vision extraction, you might want to test with a smaller document first
# to avoid high costs. Consider using a single-page sample or a smaller PDF.
selected_doc = "class_c"  # change as needed
pdf_path = TEST_DOCS[selected_doc]

if not pdf_path.exists():
    print(f"⚠️ Document not found: {pdf_path}")
else:
    print(f"✓ Selected document: {pdf_path.name}")
    print(f"  Path: {pdf_path}")
    print(f"  Size: {pdf_path.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
# Step 1: Triage and Profile

profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

print("🔍 Running Triage Agent...")
profile: DocumentProfile = triage.classify_document(pdf_path)

print("\n📋 Document Profile (summary):")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print(f"  - Estimated Cost: {profile.estimated_cost}")
print(f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  - Pages: {profile.metadata.page_count}")


In [ ]:
# Step 2: Initialize VisionExtractor

# Configure the vision model and budget
# Options for model:
#   - "openai/gpt-4o-mini" (default, cost-effective)
#   - "google/gemini-pro-vision" (alternative)
#   - Other OpenRouter vision models

vision_extractor = VisionExtractor(
    model="openai/gpt-4o-mini",  # Change to your preferred model
    max_cost_usd=0.50,  # Maximum spend per document
    cost_per_1k_tokens_usd=0.002,  # Adjust based on your model's pricing
)

print("✓ Initialized VisionExtractor")
print(f"  - Model: {vision_extractor.model}")
print(f"  - Max Cost: ${vision_extractor.budget.max_cost_usd:.2f}")
print(f"  - Cost per 1K tokens: ${vision_extractor.budget.cost_per_1k_tokens_usd:.4f}")

print("\n🔍 Checking if VisionExtractor can handle this profile...")
can_handle = vision_extractor.can_handle(profile)
print(f"  - Can Handle: {can_handle}")

if not can_handle:
    print("\n⚠️ Note: VisionExtractor typically handles scanned_image origin or")
    print("   documents marked as needs_vision_model. This profile may be better")
    print("   suited for FastTextExtractor or LayoutExtractor.")


In [ ]:
# Step 3: Confidence and Cost Estimation

print("📊 Calculating vision extraction confidence score...")
vision_conf = vision_extractor.confidence_score(str(pdf_path))
print(f"  - Confidence: {vision_conf:.4f} ({vision_conf*100:.1f}%)")

print("\n💰 Estimating vision extraction cost...")
vision_cost = vision_extractor.cost_estimate(str(pdf_path))
print(f"  - Total Estimated Cost: ${vision_cost['total_cost_usd']:.4f}")
print(f"  - Cost per Page: ${vision_cost['cost_per_page']:.4f}")
print(f"  - Budget Remaining: ${vision_extractor.budget.remaining_budget_usd:.4f}")

if vision_cost['total_cost_usd'] > vision_extractor.budget.max_cost_usd:
    print(f"\n⚠️ WARNING: Estimated cost (${vision_cost['total_cost_usd']:.4f}) exceeds")
    print(f"   budget limit (${vision_extractor.budget.max_cost_usd:.2f}). Extraction")
    print("   will stop when budget is exhausted.")


In [ ]:
# Step 4: Run Vision Extraction

print("📄 Extracting with VisionExtractor (OpenRouter)...")
print("   This may take several minutes depending on document size and API latency.")
print("   Progress will be logged as each page is processed.\n")

import time
start_time = time.time()

try:
    extracted = vision_extractor.extract(str(pdf_path))
    
    elapsed = time.time() - start_time
    print(f"\n✓ Extraction completed in {elapsed:.2f} seconds ({elapsed/60:.1f} minutes)")
    
    print("\n📊 Extraction Summary (Vision Model → ExtractedDocument):")
    print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
    print(f"  - Tables: {len(extracted.tables)}")
    print(f"  - Figures: {len(extracted.figures)}")
    print(f"  - Reading Order indices: {len(extracted.reading_order)}")
    
    print("\n💰 Budget Usage:")
    print(f"  - Total Prompt Tokens: {vision_extractor.budget.total_prompt_tokens:,}")
    print(f"  - Total Completion Tokens: {vision_extractor.budget.total_completion_tokens:,}")
    print(f"  - Total Cost: ${vision_extractor.budget.total_cost_usd:.4f}")
    print(f"  - Remaining Budget: ${vision_extractor.budget.remaining_budget_usd:.4f}")
    
except Exception as e:
    print(f"\n✗ Extraction failed: {e}")
    print("\nCommon issues:")
    print("  - OPENROUTER_API_KEY not set or invalid")
    print("  - pdf2image/poppler not installed")
    print("  - Network/API errors (check OpenRouter status)")
    print("  - Budget exhausted")
    raise


In [ ]:
# Step 5: Inspect Sample Tables and Figures

print("📊 Sample Tables (first 3):")
print("=" * 80)
for i, table in enumerate(extracted.tables[:3], start=1):
    print(f"\n[Table {i}] Page {table.page_num}")
    print(f"  Headers: {table.headers}")
    if table.rows:
        print(f"  First row: {table.rows[0]}")
        print(f"  Total rows: {len(table.rows)}")
    print(f"  BBox: ({table.bbox.x0:.1f}, {table.bbox.y0:.1f}) → ({table.bbox.x1:.1f}, {table.bbox.y1:.1f})")

print("\n🖼️ Sample Figures (first 3):")
print("=" * 80)
for i, fig in enumerate(extracted.figures[:3], start=1):
    print(f"\n[Figure {i}] Page {fig.page_num}")
    print(f"  Caption: {fig.caption or '(none)'}")
    print(f"  BBox: ({fig.bbox.x0:.1f}, {fig.bbox.y0:.1f}) → ({fig.bbox.x1:.1f}, {fig.bbox.y1:.1f})")

print("\n📝 Sample Text Blocks (first 3):")
print("=" * 80)
for i, block in enumerate(extracted.text_blocks[:3], start=1):
    print(f"\n[Text Block {i}] Page {block.page_num}")
    preview = block.content[:100] + "..." if len(block.content) > 100 else block.content
    print(f"  Content: {preview}")
    print(f"  BBox: ({block.bbox.x0:.1f}, {block.bbox.y0:.1f}) → ({block.bbox.x1:.1f}, {block.bbox.y1:.1f})")


In [ ]:
# Step 6: Summary

print("✅ VisionExtractor (OpenRouter) Test Summary:")
print("=" * 80)
print(f"\nDocument: {pdf_path.name}")
print("\nProfile:")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print(f"  - Pages: {profile.metadata.page_count}")

print("\nStrategy:")
print(f"  - Strategy: {vision_extractor.name}")
print(f"  - Model: {vision_extractor.model}")
print(f"  - Can Handle: {can_handle}")
print(f"  - Confidence: {vision_conf:.2%}")

print("\nCost:")
print(f"  - Estimated Cost: ${vision_cost['total_cost_usd']:.4f}")
print(f"  - Actual Cost: ${vision_extractor.budget.total_cost_usd:.4f}")
print(f"  - Budget Limit: ${vision_extractor.budget.max_cost_usd:.2f}")
print(f"  - Remaining: ${vision_extractor.budget.remaining_budget_usd:.4f}")

print("\nExtraction Results:")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")
print(f"  - Total Elements: {len(extracted.text_blocks) + len(extracted.tables) + len(extracted.figures)}")
